In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import sys

outputs_dir = Path("outputs")
csv_files = list(outputs_dir.glob("*_memory_stats.csv"))

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  - {f.name}")

if len(csv_files) == 0:
    print("No CSV files found in outputs/ directory!")
    sys.exit(0)

print("\n" + "="*60)

cmap = plt.cm.Set3
component_names = ['Insert Alloc', 'Retag', 'Eval Args', 'Prov GC', 'Tree Transition', 'Other Miri', 'Non-Miri']
color_map = {name: cmap(i) for i, name in enumerate(component_names)}

for csv_path in csv_files:
    project = csv_path.stem.replace("_memory_stats", "")
    
    print(f"\n{'='*60}")
    print(f"PROCESSING: {project.upper()}")
    print(f"{'='*60}\n")
    
    df = pd.read_csv(csv_path)
    
    # Convert bytes to MB
    byte_cols = [c for c in df.columns if 'bytes' in c]
    for col in byte_cols:
        df[f'{col}_mb'] = df[col] / (1024 * 1024)
    
    print(f"Total test cases: {len(df)}")
    print(f"Configs: {df['config'].unique().tolist()}")
    
    print(f"\n{'='*60}")
    print("AVERAGES BY CONFIG")
    print('='*60)
    
    configs = sorted(df['config'].unique())
    summary = []
    plot_configs = [c for c in configs if c != 'default']

    for cfg in plot_configs:
        cfg_df = df[df['config'] == cfg]
        
        data = {
            'config': cfg,
            'count': len(cfg_df),
            'total_mb': cfg_df['total_bytes_mb'].mean(),
            'insert_alloc_mb': cfg_df['insert_allocation_bytes_mb'].mean(),
            'retag_mb': cfg_df['b_retag_bytes_mb'].mean() if 'b_retag_bytes_mb' in cfg_df.columns else 0,
            'eval_args_mb': cfg_df['eval_callee_and_args_bytes_mb'].mean(),
            'prov_gc_mb': cfg_df['provenance_gc_bytes_mb'].mean(),
            'tree_transition_mb': cfg_df['perform_transition_bytes_mb'].mean() if 'perform_transition_bytes_mb' in cfg_df.columns else 0,
            'miri_mb': cfg_df['mirimachine_bytes_mb'].mean(),
        }
        
        tracked = (data['insert_alloc_mb'] + data['retag_mb'] + 
                   data['eval_args_mb'] + data['prov_gc_mb'] + data['tree_transition_mb'])
        data['other_mb'] = data['miri_mb'] - tracked
        data['non_miri_mb'] = data['total_mb'] - data['miri_mb']
        
        summary.append(data)
    
    summary_df = pd.DataFrame(summary)
    
    print("\n### Memory Usage (MB and %) ###")
    for cfg in plot_configs:
        row = summary_df[summary_df['config'] == cfg].iloc[0]
        total = row['total_mb']
        
        print(f"\n{cfg.upper()}:")
        print(f"  Insert Allocation:  {row['insert_alloc_mb']:8.2f} MB ({row['insert_alloc_mb']/total*100:5.1f}%)")
        print(f"  Retag:              {row['retag_mb']:8.2f} MB ({row['retag_mb']/total*100:5.1f}%)")
        print(f"  Eval Args:          {row['eval_args_mb']:8.2f} MB ({row['eval_args_mb']/total*100:5.1f}%)")
        print(f"  Provenance GC:      {row['prov_gc_mb']:8.2f} MB ({row['prov_gc_mb']/total*100:5.1f}%)")
        if row['tree_transition_mb'] > 0:
            print(f"  Tree Transition:    {row['tree_transition_mb']:8.2f} MB ({row['tree_transition_mb']/total*100:5.1f}%)")
        print(f"  Other MiriMachine:  {row['other_mb']:8.2f} MB ({row['other_mb']/total*100:5.1f}%)")
        print(f"  Total MiriMachine:  {row['miri_mb']:8.2f} MB ({row['miri_mb']/total*100:5.1f}%)")
        print(f"  Non-Miri:           {row['non_miri_mb']:8.2f} MB ({row['non_miri_mb']/total*100:5.1f}%)")
        print(f"  TOTAL:              {total:8.2f} MB")
    
    print(f"\n{'='*60}")
    print("GENERATING PIE CHARTS")
    print('='*60)
    
    if len(plot_configs) == 0:
        print("No configs with MiriMachine data to plot!")
    else:
        fig, axes = plt.subplots(1, len(plot_configs), figsize=(6*len(plot_configs), 6))
        if len(plot_configs) == 1:
            axes = [axes]
            
        fig.suptitle(f'{project.upper()}', fontsize=14)
        
        for idx, cfg in enumerate(plot_configs):
            row = summary_df[summary_df['config'] == cfg].iloc[0]
            
            labels = []
            sizes = []
            colors = []
            
            components = [
                ('Insert Alloc', row['insert_alloc_mb']),
                ('Retag', row['retag_mb']),
                ('Eval Args', row['eval_args_mb']),
                ('Prov GC', row['prov_gc_mb']),
                ('Tree Transition', row['tree_transition_mb']),
                ('Other Miri', row['other_mb']),
                ('Non-Miri', row['non_miri_mb'])
            ]
            
            for label, size in components:
                if size > 0 and not np.isnan(size):
                    labels.append(label)
                    sizes.append(size)
                    colors.append(color_map[label])
            
            if not sizes or all(s == 0 for s in sizes):
                print(f"Skipping {cfg} - no data")
                continue
            
            ax = axes[idx]
            ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax.set_title(f'{cfg.upper()}\n{row["total_mb"]:.1f} MB')
        
        plt.tight_layout()
        pie_path = outputs_dir / f'{project}_pies.png'
        plt.savefig(pie_path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"Saved: {pie_path}")

Found 3 CSV files:
  - socket2-outputs_memory_stats.csv
  - libc-outputs_memory_stats.csv
  - getrandom-outputs_memory_stats.csv


PROCESSING: SOCKET2-OUTPUTS

Total test cases: 30
Configs: ['miri', 'miri-tree', 'default']

AVERAGES BY CONFIG

### Memory Usage (MB and %) ###

MIRI:
  Insert Allocation:    216.47 MB ( 46.1%)
  Retag:                 25.26 MB (  5.4%)
  Eval Args:             38.40 MB (  8.2%)
  Provenance GC:          0.24 MB (  0.1%)
  Other MiriMachine:     58.45 MB ( 12.5%)
  Total MiriMachine:    338.82 MB ( 72.2%)
  Non-Miri:             130.65 MB ( 27.8%)
  TOTAL:                469.47 MB

MIRI-TREE:
  Insert Allocation:    203.06 MB ( 36.0%)
  Retag:                 41.87 MB (  7.4%)
  Eval Args:             38.88 MB (  6.9%)
  Provenance GC:          1.74 MB (  0.3%)
  Tree Transition:       72.39 MB ( 12.8%)
  Other MiriMachine:     73.50 MB ( 13.0%)
  Total MiriMachine:    431.45 MB ( 76.5%)
  Non-Miri:             132.43 MB ( 23.5%)
  TOTAL:                56